# Unit Commitment

Unit commitment adds the on/off decision of thermal units, with their start-up and shut-down behaviour, as **binary** variables. This turns the problem into a mixed-integer program.

We start from the 9-node case, ignore investments, and turn on binary commitment for a few thermal units.

## 1. Set up a working copy of the case

In [1]:
import os, shutil
import pandas as pd

DIR = "work_UC"          # parent folder that will hold the case
CaseName = "9n"      # we reuse the 9-node case from notebook 01

if os.path.exists(DIR):
    shutil.rmtree(DIR)
shutil.copytree(CaseName, os.path.join(DIR, CaseName))

# A coarse time resolution keeps the run fast for this tutorial.
param = os.path.join(DIR, CaseName, "oT_Data_Parameter_9n.csv")
df = pd.read_csv(param)
df.loc[:, "TimeStep"] = 168
df.to_csv(param, index=False)
print("Working copy of the 9n case is ready in", DIR)

Working copy of the 9n case is ready in work_UC


## 2. Activate binary unit commitment

Two things are needed: set `IndBinGenOperat = 1` in `oT_Data_Option`, and mark the units that should be committed with `BinaryCommitment = 'Yes'` in `oT_Data_Generation`. Here we commit the gas units; committing every unit would make the problem much slower to solve.

In [2]:
opt = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"))
opt.loc[0, ["IndBinGenInvest", "IndBinGenRetirement", "IndBinNetInvest"]] = 2
opt.loc[0, "IndBinGenOperat"] = 1
opt.to_csv(os.path.join(DIR, CaseName, "oT_Data_Option_9n.csv"), index=False)

gen = pd.read_csv(os.path.join(DIR, CaseName, "oT_Data_Generation_9n.csv"))
committed = ["OCGT_1", "OCGT_2", "OCGT_3", "CCGT_1", "CCGT_2"]
gen.loc[gen[gen.columns[0]].isin(committed), "BinaryCommitment"] = "Yes"
gen.to_csv(os.path.join(DIR, CaseName, "oT_Data_Generation_9n.csv"), index=False)
gen.loc[gen[gen.columns[0]].isin(committed), [gen.columns[0], "Technology", "BinaryCommitment"]]

/var/folders/sw/46j89ccx613gt8sh72tlvl1w0000gn/T/ipykernel_45389/1979044361.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Yes' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  gen.loc[gen[gen.columns[0]].isin(committed), "BinaryCommitment"] = "Yes"


,Generator,Technology,BinaryCommitment
5,CCGT_1,Gas,Yes
6,CCGT_2,Gas,Yes
9,OCGT_1,Gas,Yes
10,OCGT_2,Gas,Yes
11,OCGT_3,Gas,Yes


## 3. Run the model

This solve is a mixed-integer program, so it takes longer than the economic dispatch.

In [3]:
from openTEPES.openTEPES import openTEPES_run

model = openTEPES_run(DIR, CaseName, "appsi_highs", "Yes", "No")
print("Total system cost [MEUR]:", round(model.vTotalSCost(), 3))

Input data                             ****
Reading the CSV files                  ...  0 s
Reading    input data                  ...  0 s


Setting up input data                  ...  0 s
Setting up variables                   ...  0 s
Total cost o.f.      model formulation ****
Investment elec      model formulation ****
Period 2030, Scenario sc01, Stage st1
Generation oper o.f. model formulation ****
Investment & operation var constraints ****
Inertia, oper resr, demand constraints ****
Storage   scheduling       constraints ****
Unit commitment            constraints ****
Ramp and min up/down time  constraints ****
Network    switching model constraints ****
Network    operation model constraints ****
Problem solving                        #### 1


Termination condition:  optimal
Problem solving with fixed investments #### 1


  Total system                 cost [MEUR]  200.02029473230576  Constraints 5880  Variables 7285  Seconds 12
***** Period: 2030, Scenario: sc01, Stage: st1 ******
  Total generation  investment cost [MEUR]  0
  Total generation  retirement cost [MEUR]  0
  Total reservoir   investment cost [MEUR]  0.0
  Total network     investment cost [MEUR]  0.0
  Total H2   pipe   investment cost [MEUR]  0.0
  Total heat pipe   investment cost [MEUR]  0.0
  Total generation  operation  cost [MEUR]  200.01868831301874
  Total consumption operation  cost [MEUR]  0.0001391081034874847
  Total emission               cost [MEUR]  0.0
  Total network losses penalty cost [MEUR]  0.0014673111834946875
  Total reliability electr     cost [MEUR]  0.0
Writing            investment results  ...  0 s
Writing          cost summary results  ...  0 s
Writing           KPI summary results  ...  0 s
Writing elect network summary results  ...  0 s
Writing           reliability indexes  ...  0 s
Writing           flex

/private/tmp/claude-501/-Users-philias-ai-research-repos-openTEPES-tutorial/8f6d4ac5-9ff1-45bc-8a3c-5cc562abe3f1/scratchpad/venv312/lib/python3.12/site-packages/altair/utils/core.py:264: UserWarning: I don't know how to infer vegalite type from 'empty'.  Defaulting to nominal.
  warnings.warn(


Writing  generation operation results  ...  0 s
Writing         ESS operation results  ...  0 s
Writing elect netwk operation results  ...  0 s
Writing  marginal information results  ...  0 s


Writing              economic results  ...  0 s
Plotting electricity network     maps  ...  0 s
Total system cost [MEUR]: 200.02


## 4. Read a result

In [4]:
tech = pd.read_csv(os.path.join(DIR, CaseName, "oT_Result_TechnologyGeneration_9n.csv"))
tech.head()

,Period,Scenario,LoadLevel,Coal,ESS,Gas,Nuclear,Oil,RES
0,2030,sc01,01-07 23:00:00+01:00,43.543542,0.000000,380.301572,677.873320,0.0,82.753920
1,2030,sc01,01-14 23:00:00+01:00,108.858855,0.000000,314.138926,677.936059,0.0,139.523137
2,2030,sc01,01-21 23:00:00+01:00,108.858855,13.244153,259.744070,665.579436,0.0,196.289061
3,2030,sc01,01-28 23:00:00+01:00,108.858855,0.000000,305.896854,678.021975,0.0,231.980982
4,2030,sc01,02-04 23:00:00+01:00,43.543542,1.455992,246.578699,676.527186,0.0,320.184151
